# Titanic Baseline Pipeline

Goal: build a first simple ML pipeline for Titanic survival prediction.

This notebook includes:
- loading train and test data
- selecting baseline features
- preprocessing numeric and categorical features
- training a Logistic Regression model
- checking validation accuracy

## 1. Imports and data loading

We imported the required libraries and loaded the train and test datasets from the local data folder.


In [109]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.base import BaseEstimator
from sklearn.base import TransformerMixin

train = pd.read_csv("../../../data/titanic/train.csv")
test = pd.read_csv("../../../data/titanic/test.csv")

display(train.head())
display(test.head())

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


## 2. Select baseline features

We selected a basic set of features for the first baseline model.

The selected features include passenger class, sex, age, family-related columns, fare, and port of embarkation.

In [110]:
def add_title_helper(df):
    df = df.copy()

    # Extract passenger title from the Name column
    df["Title"] = df["Name"].str.extract(r",\s*([^\.]+)\.", expand=False)

    # Group equivalent titles to make age imputation more stable
    # Mlle and Ms are similar to Miss, Mme is similar to Mrs
    df["Title"] = df["Title"].replace({
        "Mlle": "Miss",
        "Ms": "Miss",
        "Mme": "Mrs"
    })

    # Return the dataframe with the helper Title column
    return df


# Add the helper Title column to both train and test datasets
train = add_title_helper(train)
test = add_title_helper(test)

In [111]:
features = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked", "Title"]


X = train[features]
y = train["Survived"]

X_test = test[features]

display(X.head())
display(X_test.head())

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Title
0,3,male,22.0,1,0,7.2500,S,Mr
1,1,female,38.0,1,0,71.2833,C,Mrs
2,3,female,26.0,0,0,7.9250,S,Miss
3,1,female,35.0,1,0,53.1000,S,Mrs
4,3,male,35.0,0,0,8.0500,S,Mr


,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Title
0,3,male,34.5,0,0,7.8292,Q,Mr
1,3,female,47.0,1,0,7.0000,S,Mrs
2,2,male,62.0,0,0,9.6875,Q,Mr
3,3,male,27.0,0,0,8.6625,S,Mr
4,3,female,22.0,1,1,12.2875,S,Mrs


## 3. Train-validation split

We split the training data into training and validation parts to evaluate the baseline model before making Kaggle predictions.

In [112]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

display(X_train.shape)
display(X_valid.shape)
display(y_train.shape)
display(y_valid.shape)

(712, 8)

(179, 8)

(712,)

(179,)

## 4. Numeric and categorical preprocessing


We used passenger titles as helper information to fill missing age values more accurately.

For `Age`, we used median age by passenger title and class.
For `Fare`, we used median fare by passenger class.
For `Embarked`, we used the most frequent value from the training part.

After missing values were filled, we removed the helper `Title` column from the model input

In [113]:
age_medians = X_train.groupby(["Title", "Pclass"])["Age"].median()
global_age_median = X_train["Age"].median()

fare_medians = X_train.groupby("Pclass")["Fare"].median()
global_fare_median = X_train["Fare"].median()

embarked_mode = X_train["Embarked"].mode()[0]

In [114]:
def smart_fill_missing_values(df):
    # Create a copy to avoid changing the original dataframe directly
    df = df.copy()

    # Fill Age using median age by Title and Pclass
    age_values = df.set_index(["Title", "Pclass"]).index.map(age_medians)
    df["Age"] = df["Age"].fillna(pd.Series(age_values, index=df.index))

    # If some Age values are still missing, use the global median age
    df["Age"] = df["Age"].fillna(global_age_median)

    # Fill Fare using median fare by Pclass
    df["Fare"] = df["Fare"].fillna(df["Pclass"].map(fare_medians))

    # If some Fare values are still missing, use the global median fare
    df["Fare"] = df["Fare"].fillna(global_fare_median)

    # Fill Embarked using the most frequent value from the training part
    df["Embarked"] = df["Embarked"].fillna(embarked_mode)

    return df


X_train = smart_fill_missing_values(X_train)
X_valid = smart_fill_missing_values(X_valid)
X_test = smart_fill_missing_values(X_test)

In [115]:
print("X_train missing values:")
print(X_train.isna().sum())

print("\nX_valid missing values:")
print(X_valid.isna().sum())

print("\nX_test missing values:")
print(X_test.isna().sum())

X_train missing values:
Pclass      0
Sex         0
Age         0
SibSp       0
Parch       0
Fare        0
Embarked    0
Title       0
dtype: int64

X_valid missing values:
Pclass      0
Sex         0
Age         0
SibSp       0
Parch       0
Fare        0
Embarked    0
Title       0
dtype: int64

X_test missing values:
Pclass      0
Sex         0
Age         0
SibSp       0
Parch       0
Fare        0
Embarked    0
Title       0
dtype: int64


In [116]:
# Remove Title because it was used only as a helper feature for imputation
X_train_model = X_train.drop(columns=["Title"])
X_valid_model = X_valid.drop(columns=["Title"])
X_test_model = X_test.drop(columns=["Title"])

## 5. Preprocessing and model pipeline

We removed the helper `Title` column before model training.

Then we built a pipeline for preprocessing and Logistic Regression.
Numeric features are scaled with standard scaling.
Categorical features are encoded with one-hot encoding.

In [117]:
num_cols = ["Pclass", "Age", "SibSp", "Parch", "Fare"]
cat_cols = ["Sex", "Embarked"]

num_pipeline = Pipeline([
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline([
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

## 6. Train and evaluate the baseline model

We trained the baseline Logistic Regression model on the training data.

Then we evaluated it on the validation data using accuracy.

In [118]:
pipeline.fit(X_train_model, y_train)

y_valid_pred = pipeline.predict(X_valid_model)

valid_accuracy = accuracy_score(y_valid, y_valid_pred)

print(f"Validation accuracy: {valid_accuracy:.4f}")

Validation accuracy: 0.8156


## 7. Create submission file

We used the trained baseline pipeline to make predictions for the Kaggle test dataset.

Then we saved the predictions in the required submission format.

In [119]:
test_predictions = pipeline.predict(X_test_model)

submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Survived": test_predictions
})

display(submission.head())
display(submission.shape)

,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1


(418, 2)

In [120]:
submission.to_csv("../../../submissions/titanic/baseline_logreg.csv", index=False)

## Baseline result

The baseline Logistic Regression model achieved a validation accuracy of 0.8156.

We used simple baseline features, smart missing value imputation, standard scaling for numeric features, one-hot encoding for categorical features, and Logistic Regression as the first model.

The final predictions were saved as `baseline_logreg.csv`.